# Single run: Taunton and Worcester gain MT

This notebook generates the results.

Other notebooks look at what's going on in each step.

## Scenarios

Run four scenarios: base with no bias\* and with a 10\** travel bias, and repeat with added MT available at Taunton and Worcester. Compare the results between the base and Taunton+Worcester scenario for matching time biases.

\* no bias means a 0.1 minute bias. If there are two nearest units, the 0.1 minute travel bias means that the MT unit is favoured. If there are multiple or no MT units, then the unit picked is arbitrarily the first alphabetically by postcode.

\** 10 minutes travel bias means 10.1 minutes so that MT units are favoured.


## Cohort assumptions

+ MT rate of 10%.
+ LVO rate of 30% of stroke population.
+ Two options for transfer loss rate:
  + 1.0 (everyone has a transfer for thrombectomy), and
  + 0.67 (67% of patients have a transfer for thrombectomy).
+ Admission numbers are projections for 2025. 


## Method

Initially calculate geography and basic outcome results at the LSOA-level. These basic outcomes are for separate occlusion types, nLVO and LVO, and treatment types, IVT only, MT only, and both IVT and MT. Then combine outcomes and calculate numbers of thrombectomy using the cohort assumptions.

Then combine results, gathering patients by the combination of four units, their nearest and transfer unit in the base scenario and in the Taunton+Worcester scenario. This lets us capture the movement of patient admissions when new services become available, e.g. XX new patients who go to Taunton in the new scenario would have gone to Dorchester in the base case.

Finally, combine results by single unit. For the base case, ignore the unit choices in the Taunton+Worcester scenario and vice-versa. Do separate calculations by nearest unit and by transfer unit.

  
### List of steps

Save an output file after each of these steps:

1. Define scenario parameters
1. Calculate geography and basic outcomes by LSOA.
2. Combine outcomes and calculate admissions numbers using the cohort assumptions.
3. Calculate the difference between the Taunton-Worcester scenario and base.
4. Find unit combinations.
5. Combine the differences for each unit combination across base and Taunton+Worcester.
6. Combine the differences for each unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.add_net_benefit import calculate_geo_and_outcomes, \
    calculate_cohort_outcomes_admissions
from classes.calc_scenario import build_scenario_parameters, \
    load_scenario_parameters, load_geo_cohort_data, gather_data, \
    calculate_changes
from classes.calc_by_unit import gather_unit_combos, main_unit_combos, \
    main_unit_single_all_unit_types

In [2]:
# Directories:
dir_data = 'data'
dir_output = 'output'

# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
file_unit_combos = 'unit_combos.csv'

# Output files for scenario results from each step.
# Step 1:
file_scen_params = 'scenario_parameters_taunworc.csv'
# Step 2:
file_geo_outcomes = 'results_geo_outcomes_'  # + scenario.csv
# Step 3:
file_full_results = 'results_full_'  # + scenario.csv
# Step 4:
file_results_comparison = 'results_full_comparison_taunworc_base'
# Step 5:
file_unit_combos = 'unit_combos.csv'
# Step 6:
file_comparison_unit_combo = 'comparison_by_unit_combo_base_taunworc'
# Step 7:
file_comparison_nearest_unit = 'comparison_by_nearest_unit_base_taunworc'
file_comparison_transfer_unit = 'comparison_by_transfer_unit_base_taunworc'

## Load data

### Hospital reference

In [3]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [4]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [5]:
print(len(df_hospitals))

131


In [6]:
print(len(df_hospitals.index) == len(df_hospitals.index.unique()))

True


## 1. Define scenario parameters

In [7]:
# --- Travel ---
# Allow for a travel bias time (minutes) to MT centres
dict_mt_travel_bias = {
    '': 0.1,
    '_bias10min': 10.1,
    '_bias15min': 15.1
}
s_biases = list(dict_mt_travel_bias.keys())

# --- New MT units ---
# Base case: no new units.
new_mt_units_base = []  # empty on purpose
# Main case: Taunton and Worcester are both new.
new_mt_units_main = ['TA15DA', 'WR51DD']
# Gather sets of new units:
options_new_units = [new_mt_units_base, new_mt_units_main]

# Scenario names to match sets of new units:
scenarios = ['base', 'taunworc']

### Build scenarios

For each combination of a set of new units and a proportion transferred in drip-and-ship, create scenario parameters.

In [8]:
df_scen_full = build_scenario_parameters(
        scenarios,
        options_new_units,
        dict_mt_travel_bias
    )
# Save results to file:
p = os.path.join(dir_output, file_scen_params)
df_scen_full.to_csv(p, index=False)
print(f'Saved file: {p}')

Saved file: output/scenario_parameters_taunworc.csv


Load file back in:

In [9]:
p = os.path.join(dir_output, file_scen_params)
df_scen_full = load_scenario_parameters(p)

Check scenario parameters:

In [10]:
display(df_scen_full.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
0,base,base,[],0.1
1,base_bias10min,base_bias10min,[],10.1
2,base_bias15min,base_bias15min,[],15.1
3,taunworc,taunworc,"[TA15DA, WR51DD]",0.1
4,taunworc_bias10min,taunworc_bias10min,"[TA15DA, WR51DD]",10.1
5,taunworc_bias15min,taunworc_bias15min,"[TA15DA, WR51DD]",15.1


Pick out unique geography scenario names:

In [11]:
scenarios_geo = df_scen_full['scenario_geoprocess_name'].tolist()

In [12]:
print(scenarios_geo)
print(scenarios)
print(s_biases)

['base', 'base_bias10min', 'base_bias15min', 'taunworc', 'taunworc_bias10min', 'taunworc_bias15min']
['base', 'taunworc']
['', '_bias10min', '_bias15min']


### Cohort parameters

Fixed proportion of patients have LVO (rest have nLVO), and options for proportion transferred for MT are gathered in the same results file.

In [13]:
prop_lvo = 0.3

dict_transferloss = {
    '': 1.0,
    '_transferloss0!67': 0.67
}
options_prop_lvo_dripship_treated_with_mt = list(dict_transferloss.values())
# Column names for transfer rates:
strings_tl = list(dict_transferloss.keys())

# MT rate only affects admissions and transfers, not outcomes:
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

## 2. Calculate geography and basic outcomes by LSOA

For each geography scenario, calculate the new geography and basic outcomes data and save to file as `results_geo_outcomes_{scenario_geo_name}.csv`

In [14]:
for scen_geo_name, df in df_scen_full.groupby('scenario_geoprocess_name'):
    # These parameters should be the same for all df in group:
    mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
    new_mt_units = df.iloc[0]['new_mt_units']
    print(scen_geo_name, mt_travel_bias, new_mt_units)

    # Shared geography and outcome results:
    results = calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
    )
    # Round outcome results:
    cols_mrs = [c for c in results.columns if (
                ('mrs' in c) | ('improvement' in c))]
    cols_util = [c for c in results.columns if ('util' in c)]
    results[cols_mrs] = results[cols_mrs].round(3)
    results[cols_util] = results[cols_util].round(5)

    # Save results:
    f = f'results_geo_outcomes_{scen_geo_name}.csv'
    p = os.path.join(dir_output, f)
    results.to_csv(p)
    print(f'Saved file: {p}')

base 0.1 []
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Saved file: output/results_geo_outcomes_base.csv
base_bias10min 10.1 []
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Saved file: output/results_geo_outcomes_base_bias10min.csv
base_bias15min 15.1 []
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Saved file: output/results_geo_outcomes_base_bias15min.csv
taunworc 0.1 ['TA15DA', 'WR51DD']
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Saved file: output/results_geo_outcomes_taunworc.csv
taunworc_bias10min 10.1 ['TA15DA', 'WR51DD']
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Saved file: output/results_geo_outcomes_taunworc_bias10min.csv


## 3. Combine outcomes and admissions by cohort

For each geography scenario, combine outcomes by cohort (proportion LVO and transferred for MT but __not__ MT rate - assume everybody is treated), and calculate transfer admissions numbers (using MT rate and proportion transferred for MT). Then calculate statistics by LSOA and by unit.

Save these files:
+ `results_full_{scenario_name}.csv`
+ `results_stats_{scenario_name}.csv`
+ `results_stats_by_unit_{scenario_name}.csv`


In [15]:
for scenario in scenarios_geo:
    # Load data:
    file_geo_outcomes_here = file_geo_outcomes + f'{scenario}.csv'
    p = os.path.join(dir_output, file_geo_outcomes_here)
    df_geo_outcomes = pd.read_csv(p, index_col='LSOA')
    cols_start = df_geo_outcomes.columns

    results = calculate_cohort_outcomes_admissions(
        df_geo_outcomes,
        options_prop_lvo_dripship_treated_with_mt,
        col_admissions,
        mt_rate,
        prop_lvo,
    )
    cols_new = [c for c in results.columns if c not in cols_start]
    
    # Save results:
    f = f'results_full_{scenario}.csv'
    p = os.path.join(dir_output, f)
    results[cols_new].to_csv(p)
    print(f'Saved file: {p}')

/tmp/ipykernel_23165/2821290368.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


Saved file: output/results_full_base.csv


/tmp/ipykernel_23165/2821290368.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


Saved file: output/results_full_base_bias10min.csv


/tmp/ipykernel_23165/2821290368.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


Saved file: output/results_full_base_bias15min.csv


... (2 outputs left out)

/tmp/ipykernel_23165/2821290368.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


Saved file: output/results_full_taunworc_bias10min.csv


/tmp/ipykernel_23165/2821290368.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


Saved file: output/results_full_taunworc_bias15min.csv


Calculate descriptive statistics:

In [16]:
for scenario in scenarios_geo:
    # Load in data:
    p = os.path.join(dir_output, file_geo_outcomes + scenario + '.csv')
    p2 = os.path.join(dir_output, file_full_results + scenario + '.csv')
    results = load_geo_cohort_data(p, p2)
    
    # Get descriptive stats for all numeric columns
    descriptive_stats = results.describe(percentiles=[0.25, 0.5, 0.75]).T
    descriptive_stats = descriptive_stats.round(5)
    
    # Repeat descriptive stats, grouped by nearest_ivt_unit
    results_stats_by_unit = results.groupby('nearest_ivt_unit').describe().T
    # Unpack index column so two-level index becomes two columns
    results_stats_by_unit = results_stats_by_unit.reset_index()
    rename_dict = {'level_0': 'variable', 'level_1': 'statistic'}
    results_stats_by_unit = results_stats_by_unit.rename(columns=rename_dict)
    results_stats_by_unit = results_stats_by_unit.round(5)
       
    f = f'results_stats_{scenario}.csv'
    p = os.path.join(dir_output, f)
    descriptive_stats.to_csv(p)
    print(f'Saved file: {p}')
    
    f = f'results_stats_by_unit_{scenario}.csv'
    p =os.path.join(dir_output, f)
    results_stats_by_unit.to_csv(p)
    print(f'Saved file: {p}')

/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_stats_base.csv
Saved file: output/results_stats_by_unit_base.csv


/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_stats_base_bias10min.csv
Saved file: output/results_stats_by_unit_base_bias10min.csv


/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_stats_base_bias15min.csv
Saved file: output/results_stats_by_unit_base_bias15min.csv


... (2 outputs left out)

/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_stats_taunworc_bias10min.csv
Saved file: output/results_stats_by_unit_taunworc_bias10min.csv


/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_stats_taunworc_bias15min.csv
Saved file: output/results_stats_by_unit_taunworc_bias15min.csv


## 4. Calculate the difference between the Taunton-Worcester scenario and base

Gather relevant columns and calculate the differences.

Have to calculate the comparison differently for each geography setup, e.g. compare the two scenarios that both have the same travel time bias.

In [17]:
for s_bias in s_biases:
    # Load in data:
    p = os.path.join(dir_output, file_geo_outcomes + 'base' + s_bias + '.csv')
    p2 = os.path.join(dir_output, file_full_results + 'base' + s_bias + '.csv')
    df_base = load_geo_cohort_data(p, p2)

    p = os.path.join(dir_output, file_geo_outcomes + 'taunworc' + s_bias + '.csv')
    p2 = os.path.join(dir_output, file_full_results + 'taunworc' + s_bias + '.csv')
    df_taunworc = load_geo_cohort_data(p, p2)

    df = gather_data(df_base, df_taunworc)
    df = calculate_changes(df)

    # Save to file:
    p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
    df.to_csv(p)
    print(f'Saved file: {p}')

/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_full_comparison_taunworc_base.csv


/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_full_comparison_taunworc_base_bias10min.csv


/home/anna/stroke-modelling/thrombectomy_location_2026/classes/calc_scenario.py:48: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'),


Saved file: output/results_full_comparison_taunworc_base_bias15min.csv


## 5. Find unit combinations

Find all combinations of units used in all scenarios so far.

In [18]:
# Find the names of the scenario comparison results files to be loaded in:
results_paths = {}
for s_bias in s_biases:
    f = f'{file_results_comparison}{s_bias}.csv'
    p = os.path.join(dir_output, f)
    label = 'standardbias' if len(s_bias) < 1 else s_bias.replace('_', '')
    results_paths[label] = p
# Check that files and labels make sense:
print(results_paths)

df_unit_combos = gather_unit_combos(results_paths)

# Save results:
p = os.path.join(dir_output, file_unit_combos)
df_unit_combos.to_csv(p, index=True)
print(f'Saved file: {p}')

{'standardbias': 'output/results_full_comparison_taunworc_base.csv', 'bias10min': 'output/results_full_comparison_taunworc_base_bias10min.csv', 'bias15min': 'output/results_full_comparison_taunworc_base_bias15min.csv'}
Saved file: output/unit_combos.csv


## 6. Calculate unit-combo-wise differences

Aggregate differences between the Taunton-Worcester scenario and base by combination of stroke units (nearest/transfer in base/Taunton+Worcester scenarios).

Gather hospital reference data. For each unit combination, gather the names and cities of the stroke units. These will be part of the final unit-level dataframe.

In [19]:
# Load unit combinations file:
p = os.path.join(dir_output, file_unit_combos)
df_unit_combos = pd.read_csv(p)

cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_units_here = [f'{c}_{s}' for c in cols_unit_postcodes
                   for s in scenarios]
cols_hosp = ['SSNAP name', 'Hospital_City']

# Gather SSNAP names and hospital cities:
for unit_col in cols_units_here:
    # Set up names for new columns:
    t = '_transfer_unit' if 'transfer' in unit_col else ''
    s = unit_col.split('_')[-1]  # either 'base' or 'taunworc'
    cols_new = [f'{c}{t}_{s}' for c in cols_hosp]
    rename_dict = dict(zip(cols_hosp, cols_new))
    # Merge in unit names:
    df_unit_combos = pd.merge(
        df_unit_combos,
        df_hospitals[cols_hosp].rename(columns=rename_dict).reset_index(),
        left_on=unit_col,
        right_on='Postcode',
        how='left'
    ).drop('Postcode', axis='columns')

# Check results:
display(df_unit_combos.head(3))
print(df_unit_combos.columns)

,unit_combo,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min,bias15min,SSNAP name_base,Hospital_City_base,SSNAP name_taunworc,Hospital_City_taunworc,SSNAP name_transfer_unit_base,Hospital_City_transfer_unit_base,SSNAP name_transfer_unit_taunworc,Hospital_City_transfer_unit_taunworc
0,42000,BN25BE,BN25BE,BN25BE,BN25BE,1,0,0,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
1,42001,BN112DH,BN112DH,BN25BE,BN25BE,1,0,0,Worthing Hospital,Worthing,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
2,42002,CA27HY,CA27HY,NE14LP,NE14LP,1,0,0,Cumberland Infirmary,Carlisle,Cumberland Infirmary,Carlisle,Royal Victoria Infirmary,Newcastle,Royal Victoria Infirmary,Newcastle


Index(['unit_combo', 'nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc',
       'transfer_unit_base', 'transfer_unit_taunworc', 'standardbias',
       'bias10min', 'bias15min', 'SSNAP name_base', 'Hospital_City_base',
       'SSNAP name_taunworc', 'Hospital_City_taunworc',
       'SSNAP name_transfer_unit_base', 'Hospital_City_transfer_unit_base',
       'SSNAP name_transfer_unit_taunworc',
       'Hospital_City_transfer_unit_taunworc'],
      dtype='str')


Run calculations.

In [20]:
for s_bias in s_biases:
    # Load in the LSOA-level outcomes and admissions data:
    p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
    df = pd.read_csv(p, index_col=0)
    
    # Merge unit combination labels into the main dataframe:
    df = pd.merge(df, df_unit_combos, on=cols_units_here, how='left')

    # Main calculations:
    comparison_by_unit_combo = main_unit_combos(
        df,
        df_unit_combos,
        scenarios,
        col_admissions,
        cols_unit_postcodes,
        strings_tl,
        )

    # Save results with all unit combos present in df:
    p = os.path.join(dir_output, file_comparison_unit_combo + s_bias + '.csv')
    comparison_by_unit_combo.to_csv(p, index=True)
    print(f'Saved file: {p}')

Unit combos as expected: True
Saved file: output/comparison_by_unit_combo_base_taunworc.csv
Unit combos as expected: True
Saved file: output/comparison_by_unit_combo_base_taunworc_bias10min.csv
Unit combos as expected: True
Saved file: output/comparison_by_unit_combo_base_taunworc_bias15min.csv


## 7. Calculate unit-wise differences

Aggregate differences between the Taunton-Worcester scenario and base by single stroke unit.

Create four sets of results: base nearest unit, base transfer unit, Taunton+Worcester nearest unit, and Taunton+Worcester transfer unit. 

### Run calculations


In [21]:
for s_bias in s_biases:
    # Load in the LSOA-level outcomes and admissions data:
    p = os.path.join(dir_output, file_comparison_unit_combo + s_bias + '.csv')
    comparison_by_unit_combo = pd.read_csv(p, index_col=0)

    comparison_by_nearest_ivt_unit, comparison_by_transfer_unit = (
        main_unit_single_all_unit_types(
            comparison_by_unit_combo,
            scenarios,
            col_admissions
        )
    )
    
    p = os.path.join(dir_output, file_comparison_nearest_unit + s_bias + '.csv')
    comparison_by_nearest_ivt_unit.to_csv(p)
    print(f'Saved file: {p}')
    
    p = os.path.join(dir_output, file_comparison_transfer_unit + s_bias + '.csv')
    comparison_by_transfer_unit.to_csv(p)
    print(f'Saved file: {p}')

Saved file: output/comparison_by_nearest_unit_base_taunworc.csv
Saved file: output/comparison_by_transfer_unit_base_taunworc.csv
Saved file: output/comparison_by_nearest_unit_base_taunworc_bias10min.csv
Saved file: output/comparison_by_transfer_unit_base_taunworc_bias10min.csv
Saved file: output/comparison_by_nearest_unit_base_taunworc_bias15min.csv
Saved file: output/comparison_by_transfer_unit_base_taunworc_bias15min.csv


## Conclusion

All of the results files have been created.